# NPDES Data Cleaning: ECHO Facilities (Iowa)

Cleans the EPA ECHO facility registry for Iowa NPDES permittees into one tidy
row per facility, keyed on `npdes_id`, ready to join to the discharge-monitoring
(DMR), catchment, and impaired-waters (ATTAINS) tables.

**Input:**  `data/tabular/01_raw/npdes/echo-facilities-iowa.csv`
**Output:** `data/tabular/02_clean/npdes/echo-facilities-clean.csv`

Each row is one permitted facility:

| raw column             | meaning                                          | type   |
|------------------------|--------------------------------------------------|--------|
| `facility_interest_id` | ECHO facility-interest id                        | id     |
| `npdes_id`             | NPDES permit number (join key)                   | id     |
| `facility_uin`         | FRS unique identifier                            | id     |
| `facility_type_code`   | ECHO facility-type code (POF, COR, …)            | code   |
| `facility_name`        | facility name                                    | text   |
| `address`/`city`/`zip` | mailing location                                 | text   |
| `county_fips`          | county, as postal-prefixed code (e.g. `IA111`)   | code   |
| `latitude`/`longitude` | facility coordinates (NAD83)                     | deg    |
| `impaired_waters`      | `303(D) Listed` flag where present               | text   |

**Cleaning steps** — normalize ids to strings, convert the postal-prefixed
`county_fips` to a standard 5-digit numeric FIPS, truncate ZIP+4 to a 5-digit
ZIP, repair sign-flipped longitudes, turn `impaired_waters` into a boolean, then
validate the Iowa bounding box, de-duplicate on `npdes_id`, and write.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "npdes"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "npdes"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/npdes
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/npdes


## Step 1 — Load

Read every column as a string so identifiers (`facility_uin`, `zip`,
`county_fips`) keep their exact form; numeric coercion happens explicitly
below.

In [2]:
df = pd.read_csv(RAW_DIR / "echo-facilities-iowa.csv", dtype="string")
n_raw = len(df)
print(f"Loaded {n_raw:,} facilities")
print("Columns:", list(df.columns))
print("\nNulls per column:")
print(df.isna().sum().to_string())
df.head(3)

Loaded 2,216 facilities
Columns: ['facility_interest_id', 'npdes_id', 'facility_uin', 'facility_type_code', 'facility_name', 'address', 'city', 'county_fips', 'zip', 'latitude', 'longitude', 'impaired_waters']

Nulls per column:
facility_interest_id       0
npdes_id                   0
facility_uin               2
facility_type_code       257
facility_name              0
address                    1
city                       0
county_fips              203
zip                        0
latitude                  78
longitude                 78
impaired_waters         1855


,facility_interest_id,npdes_id,facility_uin,facility_type_code,facility_name,address,city,county_fips,zip,latitude,longitude,impaired_waters
0,3297591,IA0000523,110013331983,POF,"HENNIGES AUTOMOTIVE, IOWA INC.",3200 MAIN STREET,KEOKUK,IA111,526328230,40.415833,-91.404167,<NA>
1,5647021,IA0077887,110014325550,<NA>,S + S FARM,12713 120TH AVE,GIBSON,IA107,50104,41.47134,-92.37455,<NA>
2,600004546,IA0063886,110000413428,COR,"FLEETGUARD, INC.",311 NORTH PARK STREET,LAKE MILLS,IA189,50450,43.42147,-93.53952,<NA>


## Step 2 — Normalize identifiers and text

`facility_uin` arrives as a float (`1.10e11`), so cast it through a nullable
integer to drop the spurious `.0` before stringifying. Trim stray whitespace
from the free-text fields.

In [3]:
df["facility_uin"] = (
    pd.to_numeric(df["facility_uin"], errors="coerce").astype("Int64").astype("string")
)
df["facility_interest_id"] = df["facility_interest_id"].str.strip()

TEXT_COLS = ["facility_name", "address", "city", "facility_type_code"]
for c in TEXT_COLS:
    df[c] = df[c].str.strip()

print(df[["facility_interest_id", "npdes_id", "facility_uin", "facility_type_code"]].head(3))

  facility_interest_id   npdes_id  facility_uin facility_type_code
0              3297591  IA0000523  110013331983                POF
1              5647021  IA0077887  110014325550               <NA>
2            600004546  IA0063886  110000413428                COR


## Step 3 — Convert `county_fips` to a 5-digit numeric FIPS

The raw code is a postal abbreviation plus a 3-digit county code (e.g.
`IA111` = Lee County). Map the postal prefix to its 2-digit state FIPS and
concatenate, yielding the canonical 5-digit county FIPS (`19111`). A handful of
border facilities sit in neighboring states; all observed prefixes are mapped,
and the guard fails loudly if a future pull introduces an unmapped one.

In [4]:
STATE_POSTAL_TO_FIPS = {
    "IA": "19", "IL": "17", "MN": "27", "MO": "29",
    "NE": "31", "SD": "46", "WI": "55", "ND": "38",
}

postal = df["county_fips"].str[:2]
county = df["county_fips"].str[2:]
unmapped = set(postal.dropna()) - set(STATE_POSTAL_TO_FIPS)
assert not unmapped, f"unmapped county_fips state prefixes: {unmapped}"

df["county_fips"] = postal.map(STATE_POSTAL_TO_FIPS) + county
ok = df["county_fips"].dropna()
assert ok.str.fullmatch(r"\d{5}").all(), "county_fips not 5 digits"
print("State FIPS present:", sorted(df["county_fips"].dropna().str[:2].unique()))
print(f"county_fips missing: {df['county_fips'].isna().sum()}")

State FIPS present: ['17', '19']
county_fips missing: 203


## Step 4 — Truncate ZIP to 5 digits

Some rows carry a 9- or 10-character ZIP+4 (`526328230`, `52632-8230`). Keep
only the 5-digit base ZIP.

In [5]:
df["zip"] = df["zip"].str.replace("-", "", regex=False).str[:5]
assert df["zip"].dropna().str.fullmatch(r"\d{5}").all(), "ZIP not 5 digits"
print(df["zip"].str.len().value_counts(dropna=False).to_string())

zip
5    2216


## Step 5 — Repair coordinates

Cast lat/lon to float. A few rows have a **positive** longitude — a dropped
minus sign, since every Iowa facility is in the western hemisphere. Flip those
to negative, then confirm all coordinates fall inside a generous Iowa bounding
box (lat 40–44 N, lon 90–97 W).

In [6]:
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

flipped = df["longitude"] > 0
print(f"Sign-flipped longitudes repaired: {int(flipped.sum())}")
df.loc[flipped, "longitude"] = -df.loc[flipped, "longitude"]

coord = df.dropna(subset=["latitude", "longitude"])
assert coord["latitude"].between(40, 44).all(), "latitude outside Iowa bbox"
assert coord["longitude"].between(-97, -90).all(), "longitude outside Iowa bbox"
print(f"Coordinates present: {len(coord):,} / {len(df):,}")
print("lat range:", coord["latitude"].min(), "→", coord["latitude"].max())
print("lon range:", coord["longitude"].min(), "→", coord["longitude"].max())

Sign-flipped longitudes repaired: 4
Coordinates present: 2,138 / 2,216
lat range: 40.3867 → 43.50049
lon range: -96.61731 → -90.14389


## Step 6 — Boolean impaired-waters flag

`impaired_waters` is either `303(D) Listed` or null, so it is really a flag.
Replace it with a boolean `impaired_303d`.

In [7]:
print(df["impaired_waters"].value_counts(dropna=False).to_string())
df["impaired_303d"] = df["impaired_waters"].notna()
df = df.drop(columns="impaired_waters")
print(f"\n303(d)-listed facilities: {int(df['impaired_303d'].sum())}")

impaired_waters
<NA>             1855
303(D) Listed     361

303(d)-listed facilities: 361


## Step 7 — Order, de-duplicate, and write

`npdes_id` should uniquely identify a facility. Confirm it, order columns
key-first, sort, and write the tidy table.

In [8]:
ORDERED = [
    "npdes_id", "facility_interest_id", "facility_uin", "facility_name",
    "facility_type_code", "address", "city", "county_fips", "zip",
    "latitude", "longitude", "impaired_303d",
]
clean = df[ORDERED].sort_values("npdes_id").reset_index(drop=True)

dupes = clean["npdes_id"].duplicated().sum()
assert dupes == 0, f"{dupes} duplicate npdes_id rows"

out_path = CLEAN_DIR / "echo-facilities-clean.csv"
clean.to_csv(out_path, index=False)
print(f"Wrote {len(clean):,} rows × {clean.shape[1]} cols (from {n_raw:,}) to:")
print(" ", out_path.relative_to(REPO_ROOT))
clean.head()

Wrote 2,216 rows × 12 cols (from 2,216) to:
  data/tabular/02_clean/npdes/echo-facilities-clean.csv


,npdes_id,facility_interest_id,facility_uin,facility_name,facility_type_code,address,city,county_fips,zip,latitude,longitude,impaired_303d
0,IA0000035,3200010097,110036385104,"OTTUMWA WATER WORKS, CITY OF",CTG,230 TURNER DRIVE,OTTUMWA,19179,52501,41.01903,-92.41676,True
1,IA0000051,3200030608,110045323020,JOHN DEERE DUBUQUE WORKS,COR,18600 SOUTH JOHN DEERE ROAD,DUBUQUE,19061,52004,42.56591,-90.69351,True
2,IA0000060,3200015686,110000413954,JOHN DEERE WATERLOO WORKS (DRIVE TRAIN OPERATI...,COR,400 WESTFIELD AVENUE,WATERLOO,19013,50704,42.50304,-92.35298,True
3,IA0000108,3200033948,110019914011,IPL - SUTHERLAND STATION,COR,2115 E NEVADA ST,MARSHALLTOWN,19127,50158,42.05746,-92.85389,False
4,IA0000132,3200016237,110013397458,IPL - SIXTH STREET GENERATING STATION,COR,509 6TH STREET NE,CEDAR RAPIDS,19113,52401,41.98472,-91.66876,False
